# Couverture et qualité des sources d'offres d'alternance IT
## Toulouse vs Île-de-France — snapshot du 3 octobre 2026

**Question posée.** Trois sources alimentent un pipeline de veille d'offres
d'alternance / stage IT. Elles ne se valent pas : elles ne couvrent pas les mêmes
zones, ne remplissent pas les mêmes champs, et ne rejettent pas pour les mêmes
raisons. Ce notebook mesure **ce que chaque source apporte réellement** et **où
elle est muette**, pour Toulouse et l'Île-de-France.

---

### Ce que ce notebook est

Une analyse de **qualité et de couverture de sources de données**. L'objet étudié
est le pipeline de collecte, pas le marché de l'emploi.

### Ce que ce notebook n'est pas

> ⚠️ **C'est un échantillon de ma recherche, pas un échantillon du marché.**

Les 276 offres analysées sont celles que *mes* requêtes et *mes* alertes mail ont
ramenées, sur *mes* deux zones, pendant *huit jours*. Aucun chiffre ici ne décrit
le marché français de l'alternance IT. Une phrase comme « il y a plus d'offres
data en Île-de-France » serait fausse ; la phrase correcte est « ma veille a
ramené plus d'offres data en Île-de-France ».

### Périmètre volontairement exclu

| Exclu | Pourquoi |
|---|---|
| Salaires | `remuneration` renseignée sur 11 % des offres, en texte libre |
| Compétences | `skills_required` sur 38 %, `skills_optional` sur **0 %** |
| Niveau d'études | `education_required` sur **2,2 %** |
| Nuage de mots | aucune description d'offre n'est collectée |

Ces colonnes sont trop vides pour produire autre chose qu'un artefact. Leur vide
est **lui-même un résultat** : il est mesuré en partie 3, puis le sujet est clos.

### Confidentialité

Les données sources sont dans `data/private/` (exclu de git). Le notebook
**n'affiche jamais** l'intitulé, l'entreprise, l'URL ni les notes d'une offre :
ces colonnes sont supprimées au chargement (partie 1), ce qui rend la fuite
structurellement impossible plutôt que simplement déconseillée. Seuls des
agrégats sont exportés, dans `data/public/`.

---
## 1. Chargement et garde-fou de confidentialité

Le `scp` depuis le serveur a créé un sous-dossier `export/`, donc les fichiers
sont retrouvés par glob récursif plutôt que par un chemin figé : le notebook
marche quelle que soit la forme exacte de la copie.

Les colonnes sensibles sont lues une seule fois, pour en **dériver des
indicateurs booléens**, puis supprimées. Après la cellule de chargement elles
n'existent plus dans la mémoire du notebook : aucun `head()` ne peut les trahir,
y compris par accident dans une cellule ajoutée plus tard.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

RACINE = Path.cwd()
DOSSIER_PRIVE = RACINE / "data" / "private"
DOSSIER_PUBLIC = RACINE / "data" / "public"
DOSSIER_FIGURES = RACINE / "figures"
DOSSIER_PUBLIC.mkdir(parents=True, exist_ok=True)
DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)

SNAPSHOT = pd.Timestamp("2026-10-03", tz="UTC")

# Colonnes supprimees apres derivation : identifiantes ou nominatives.
COLONNES_SENSIBLES = ["title", "company", "canonical_url", "notes", "location"]


def trouver(nom_fichier):
    '''Localise un CSV sous data/private/, quel que soit le sous-dossier.'''
    trouves = sorted(DOSSIER_PRIVE.glob("**/" + nom_fichier))
    if not trouves:
        raise FileNotFoundError(
            nom_fichier + " introuvable sous " + str(DOSSIER_PRIVE)
            + ". Relancer la copie depuis le serveur (voir README)."
        )
    return trouves[0]


def lire_csv(nom_fichier):
    # utf-8-sig : les exports du serveur portent un BOM, sinon la premiere
    # colonne s'appelle 'i>>?id' au lieu de 'id'.
    return pd.read_csv(trouver(nom_fichier), encoding="utf-8-sig")


print("Fichiers trouves :")
for f in ["offers_full.csv", "source_runs.csv", "alerts.csv", "offer_actions.csv"]:
    print("  {:20s} -> {}".format(f, trouver(f).relative_to(RACINE)))

In [ ]:
# La completude (partie 3) doit porter sur TOUTES les colonnes, y compris les
# sensibles. Elle est donc calculee ici, avant la suppression ; seul le tableau
# de taux est conserve, jamais les valeurs.

_brut = lire_csv("offers_full.csv")
N_OFFRES = len(_brut)
COLONNES_ORIGINE = list(_brut.columns)


def taux_remplissage(df):
    '''% de valeurs reellement utilisables par colonne.

    Une chaine vide ou faite d'espaces compte comme manquante : pandas la
    considere remplie, un analyste non.
    '''
    out = {}
    for col in df.columns:
        s = df[col]
        if s.dtype == object or isinstance(s.dtype, pd.StringDtype):
            rempli = s.astype("string").str.strip().replace("", pd.NA).notna()
        else:
            rempli = s.notna()
        out[col] = round(100 * float(rempli.mean()), 1)
    return pd.Series(out, name="rempli_%")


COMPLETUDE_GLOBALE = taux_remplissage(_brut)
COMPLETUDE_PAR_SOURCE = pd.DataFrame({
    src: taux_remplissage(sous)
    for src, sous in _brut.groupby("source", observed=True)
})

del _brut  # les valeurs sensibles ne survivent pas a cette cellule

print("{} offres, {} colonnes.".format(N_OFFRES, len(COLONNES_ORIGINE)))
print("Completude calculee avant suppression des colonnes sensibles.")

In [ ]:
def charger_offres():
    '''Charge les offres, derive les indicateurs, puis supprime le sensible.'''
    d = lire_csv("offers_full.csv")

    for col in ["published_at", "first_seen_at", "last_seen_at", "alerted_at"]:
        d[col] = pd.to_datetime(d[col], errors="coerce", utc=True)

    notes = d["notes"].fillna("")

    # --- Indicateurs derives des notes (le texte ne sera pas conserve) -------
    # Le moteur ecrit "type de contrat incompatible : None" quand la source n'a
    # fourni AUCUN type de contrat. Ce n'est pas un refus sur criteres, c'est une
    # metadonnee absente. Cette distinction est le coeur de la partie 6.
    d["contrat_non_fourni"] = notes.str.contains("incompatible : None", regex=False)
    d["cdi_cdd_mal_classe"] = notes.str.contains("CDI/CDD mal class", regex=False)
    d["ecole_imposee"] = notes.str.contains("cole/CFA partenaire impos", regex=False)
    d["ecarte_manuellement"] = notes.str.contains("utilisateur", regex=False)
    d["entreprise_non_identifiee"] = notes.str.contains(
        "Entreprise non identifi", regex=False
    )

    # location : conservee seulement comme booleen. Elle contient parfois un nom
    # d'entreprise (bug de parsing documente en partie 4), donc du nominatif.
    d["location_renseignee"] = (
        d["location"].astype("string").str.strip().replace("", pd.NA).notna()
    )

    return d.drop(columns=COLONNES_SENSIBLES)


offres = charger_offres()

# Verification du garde-fou : aucune colonne sensible ne survit.
restantes = [c for c in COLONNES_SENSIBLES if c in offres.columns]
assert not restantes, "COLONNE SENSIBLE ENCORE PRESENTE : {}".format(restantes)
print("Garde-fou OK : title / company / canonical_url / notes / location supprimees.")
print("offres : {} lignes x {} colonnes".format(*offres.shape))

runs = lire_csv("source_runs.csv")
for col in ["started_at", "finished_at"]:
    runs[col] = pd.to_datetime(runs[col], errors="coerce", utc=True)
print("runs   : {} executions de collecte".format(len(runs)))

### Un raccord de nomenclature à faire avant toute jointure

`source_runs` et `offers_full` ne nomment pas la même source de la même façon :
le collecteur s'appelle `email` (il dépouille les alertes mail), les offres qu'il
produit sont étiquetées `linkedin` (l'origine de ces alertes). C'est la même
brique. Sans ce raccord, l'entonnoir de la partie 5 ne se relierait à aucune
offre et `linkedin` paraîtrait sortir de nulle part.

In [ ]:
COLLECTEUR_VERS_SOURCE = {"email": "linkedin"}
runs["source_offre"] = runs["source"].map(
    lambda s: COLLECTEUR_VERS_SOURCE.get(s, s)
)

# Ordres d'affichage stables, pour que les figures se lisent entre elles.
ORDRE_SOURCES = offres["source"].value_counts().index.tolist()
ORDRE_ZONES = ["toulouse", "idf", "autre", "inconnu"]

print("Collecteurs (source_runs) :", sorted(runs["source"].unique()))
print("Sources (offers_full)     :", sorted(offres["source"].unique()))
print("Apres raccord             :", sorted(runs["source_offre"].unique()))

### Style des graphiques

Un seul endroit définit l'apparence, pour que les figures se lisent comme un même
jeu. La palette catégorielle est fixée **par entité** — une source garde sa
couleur d'un graphique à l'autre — et non par rang : un lecteur qui a appris
« LinkedIn est en bleu » ne doit pas être détrompé à la figure suivante.

Les catégories qui signifient « information absente » (`zone inconnue`,
`doublons`) prennent un **gris neutre** plutôt qu'une couleur d'identité : une
donnée manquante n'est pas une modalité comme les autres.

Les effectifs sont écrits au bout de chaque barre. C'est une exigence du sujet,
et aussi la contrepartie obligatoire d'une palette dont une teinte passe sous le
seuil de contraste de 3:1 face au fond.

In [ ]:
# Palette categorielle validee : separation deutan/tritan >= 8 sur toutes les
# paires, plancher vision normale >= 15.
BLEU, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
GRIS_NEUTRE = "#767572"   # categories "information absente"
ROUGE_ALERTE = "#e34948"  # statut uniquement, jamais une serie
ENCRE, ENCRE_2 = "#0b0b0b", "#52514e"

COULEUR_SOURCE = {"linkedin": BLEU, "france_travail": ORANGE, "labonnealternance": AQUA}
COULEUR_ZONE = {"toulouse": BLEU, "idf": ORANGE, "autre": AQUA, "inconnu": GRIS_NEUTRE}

LIBELLE_SOURCE = {
    "linkedin": "LinkedIn (alertes mail)",
    "france_travail": "France Travail (API)",
    "labonnealternance": "La Bonne Alternance (API)",
}
LIBELLE_ZONE = {
    "toulouse": "Toulouse",
    "idf": "Ile-de-France",
    "autre": "Autre region",
    "inconnu": "Zone inconnue",
}
LIBELLE_METIER = {
    "fullstack": "Full stack", "data": "Data", "backend": "Backend",
    "devops": "DevOps", "mobile": "Mobile", "autre": "Autre / non classe",
}

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "savefig.facecolor": "#fcfcfb",
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"],
    "font.size": 9.5,
    "axes.titlesize": 11.5,
    "axes.titleweight": "semibold",
    "axes.titlecolor": ENCRE,
    "axes.titlelocation": "left",
    "axes.titlepad": 14,
    "axes.labelcolor": ENCRE_2,
    "axes.labelsize": 9,
    "axes.edgecolor": "#dedcd6",
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.color": ENCRE_2,
    "ytick.color": ENCRE_2,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "grid.color": "#e8e6e0",
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",   # jamais de pointilles : ca se lit comme un seuil
    "legend.frameon": False,
    "legend.fontsize": 9,
    "legend.handlelength": 0.9,
    "legend.handleheight": 0.9,
    "legend.borderpad": 0.0,
    "legend.columnspacing": 1.4,
})


def habiller(ax, titre, sous_titre=None, axe_valeurs="x"):
    '''Titre, sous-titre explicatif, grille discrete sur le seul axe des valeurs.'''
    ax.set_title(titre)
    if sous_titre:
        ax.text(0, 1.012, sous_titre, transform=ax.transAxes, ha="left", va="bottom",
                fontsize=8.6, color=ENCRE_2)
    ax.grid(axis=axe_valeurs, alpha=0.9, zorder=0)
    ax.set_axisbelow(True)
    return ax


def etiqueter(ax, conteneur, suffixe="", horizontal=True, marge=0.014):
    '''Ecrit l'effectif au bout de chaque barre, a l'exterieur de la barre.

    A l'exterieur, et non dedans : une etiquette posee dans une barre courte est
    rognee. Les barres a zero ne sont pas etiquetees.
    '''
    lim = ax.get_xlim()[1] if horizontal else ax.get_ylim()[1]
    for barre in conteneur:
        v = barre.get_width() if horizontal else barre.get_height()
        if v <= 0:
            continue
        texte = "{:,.0f}".format(v).replace(",", " ") + suffixe
        if horizontal:
            ax.text(v + lim * marge, barre.get_y() + barre.get_height() / 2,
                    texte, va="center", ha="left", fontsize=8.6, color=ENCRE_2)
        else:
            ax.text(barre.get_x() + barre.get_width() / 2, v + lim * marge,
                    texte, ha="center", va="bottom", fontsize=8.6, color=ENCRE_2)


def enregistrer(fig, nom):
    '''Sauve la figure dans figures/ : elle ne contient que des agregats.'''
    chemin = DOSSIER_FIGURES / (nom + ".png")
    fig.savefig(chemin)
    print("figure -> {}".format(chemin.relative_to(RACINE)))


print("Style charge.")

---
## 2. Contrôles de cohérence — avant toute analyse

Un chiffre sorti d'une table qu'on n'a pas vérifiée n'est pas un résultat, c'est
une supposition. Cette partie passe cinq contrôles automatiques et affiche leur
verdict. **Deux échouent**, et ce qu'ils révèlent change la lecture de tout le
reste du notebook — c'est pourquoi ils viennent en premier et non en annexe.

In [ ]:
controles = []


def controle(nom, attendu, observe, ok, consequence=""):
    controles.append({
        "controle": nom,
        "attendu": attendu,
        "observe": observe,
        "statut": "OK" if ok else "ECHEC",
        "consequence": consequence,
    })


# --- C1 : l'identite de l'entonnoir tient-elle, run par run ? --------------
# trouvees = nouvelles + doublons + rejetees. Si cette somme ne tombe pas juste,
# les compteurs du collecteur sont inutilisables et la partie 5 n'a pas de sens.
ecart_entonnoir = (
    runs["offers_found"]
    - runs[["offers_new", "offers_duplicate", "offers_rejected"]].sum(axis=1)
)
n_runs_incoherents = int((ecart_entonnoir != 0).sum())
controle(
    "C1 - identite de l'entonnoir (par run)",
    "trouvees = nouvelles + doublons + rejetees sur les 554 runs",
    "{} run(s) en ecart".format(n_runs_incoherents),
    n_runs_incoherents == 0,
    "Les compteurs de collecte sont exploitables (partie 5).",
)

# --- C2 : les offres annoncees "nouvelles" sont-elles toutes stockees ? ----
nouvelles_annoncees = runs.groupby("source_offre")["offers_new"].sum()
offres_stockees = offres["source"].value_counts()
recon = pd.DataFrame({
    "annoncees_nouvelles": nouvelles_annoncees,
    "lignes_stockees": offres_stockees,
}).fillna(0).astype(int)
recon["ecart"] = recon["lignes_stockees"] - recon["annoncees_nouvelles"]
ecart_total = int(recon["ecart"].sum())
controle(
    "C2 - reconciliation runs / table d'offres",
    "somme(offers_new) = nombre de lignes stockees",
    "{} annoncees vs {} stockees (ecart {:+d})".format(
        int(recon["annoncees_nouvelles"].sum()),
        int(recon["lignes_stockees"].sum()),
        ecart_total,
    ),
    ecart_total == 0,
    "La table d'offres a ete reinitialisee en cours de periode : voir ci-dessous.",
)

# --- C3 : un identifiant = une offre --------------------------------------
doublons_id = int(offres["id"].duplicated().sum())
controle(
    "C3 - unicite des identifiants d'offre",
    "aucun id en double",
    "{} doublon(s)".format(doublons_id),
    doublons_id == 0,
    "Les comptages d'offres ne surestiment rien.",
)

# --- C4 : les runs couvrent-ils bien la periode des offres ? ---------------
debut_runs, fin_runs = runs["started_at"].min(), runs["started_at"].max()
debut_offres, fin_offres = offres["first_seen_at"].min(), offres["first_seen_at"].max()
runs_couvrent = debut_runs <= debut_offres and fin_runs >= fin_offres
controle(
    "C4 - couverture temporelle des runs",
    "la periode des runs englobe celle des offres",
    "runs {} -> {} | offres {} -> {}".format(
        debut_runs.date(), fin_runs.date(), debut_offres.date(), fin_offres.date()
    ),
    bool(runs_couvrent),
    "Aucune offre n'arrive d'une periode non journalisee.",
)

# --- C5 : y a-t-il des imports en masse deguises en detections live ? -----
# Si 100 offres partagent la meme seconde de first_seen_at, ce n'est pas une
# detection au fil de l'eau : c'est un chargement par lot. Les delais de
# detection calcules sur ces lignes mesureraient l'import, pas la veille.
lots = offres["first_seen_at"].value_counts()
SEUIL_LOT = 25
LOTS_MASSE = lots[lots >= SEUIL_LOT]
n_en_lot = int(LOTS_MASSE.sum())
controle(
    "C5 - absence d'import en masse",
    "aucun horodatage ne porte plus de {} offres".format(SEUIL_LOT),
    "{} offre(s) dans {} lot(s)".format(n_en_lot, len(LOTS_MASSE)),
    len(LOTS_MASSE) == 0,
    "Ces lignes sont ecartees du calcul des delais (partie 8).",
)

TABLEAU_CONTROLES = pd.DataFrame(controles)
print("CONTROLES DE COHERENCE\n")
for c in controles:
    marque = "[ OK  ]" if c["statut"] == "OK" else "[ECHEC]"
    print("{} {}".format(marque, c["controle"]))
    print("        attendu : {}".format(c["attendu"]))
    print("        observe : {}".format(c["observe"]))
    print("        -> {}\n".format(c["consequence"]))

### C2 — ce que l'échec révèle

Les deux tables ne racontent pas la même histoire. Le détail par source, puis la
chronologie, expliquent pourquoi.

In [ ]:
print("Reconciliation par source :\n")
print(recon.rename(index=LIBELLE_SOURCE).to_string())

# Chronologie : des offres sont annoncees nouvelles avant la premiere offre
# effectivement presente dans la table.
runs_par_jour = (
    runs.assign(jour=runs["started_at"].dt.date)
    .groupby(["jour", "source_offre"])["offers_new"].sum().unstack(fill_value=0)
)
offres_par_jour = (
    offres.assign(jour=offres["first_seen_at"].dt.date)
    .groupby(["jour", "source"]).size().unstack(fill_value=0)
)

premier_jour_offre = offres["first_seen_at"].min().date()
avant = runs_par_jour[runs_par_jour.index < premier_jour_offre]

print("\n\nNouvelles offres annoncees AVANT la premiere offre presente en table :")
print(avant.to_string() if len(avant) else "(aucune)")
print("\n-> {} offres declarees nouvelles entre le {} et le {} n'ont laisse".format(
    int(avant.to_numpy().sum()), runs["started_at"].min().date(),
    (pd.Timestamp(premier_jour_offre) - pd.Timedelta(days=1)).date()))
print("   aucune ligne dans la table. Elles ont ete purgees.")

**Conclusion des contrôles** — trois faits à garder en tête pour la suite :

1. **Les compteurs de collecte sont fiables** (C1 passe exactement, sur 554 runs).
   L'entonnoir de la partie 5 repose sur du solide.
2. **La table d'offres a été réinitialisée en cours de période.** Le pipeline
   tourne depuis le 23 septembre, mais aucune offre antérieure au 26 septembre
   n'y subsiste. La **fenêtre d'observation réelle est donc de 8 jours**
   (26 sept → 3 oct), pas de 11. Toute phrase du type « en 11 jours de veille »
   serait fausse.
3. **Une partie des offres est entrée par lot**, pas au fil de l'eau (C5). Pour
   ces lignes, `first_seen_at` est l'heure du chargement et non l'heure de
   détection : elles sont écartées du calcul des délais en partie 8.

Aucun de ces trois points n'était visible dans les données elles-mêmes — il a
fallu confronter les deux tables. C'est exactement ce que ces contrôles servent
à attraper.

---
## 3. Complétude — le vide comme résultat

La complétude est présentée ici **comme un résultat et non comme une étape de
nettoyage**, parce que c'est elle qui décide de ce qui est analysable. Une colonne
remplie à 2 % ne se « nettoie » pas : elle se renonce.

Le tableau croise le taux de remplissage avec la source, parce que le vide n'est
pas réparti au hasard : il dépend presque entièrement de qui a fourni l'offre.

In [ ]:
# On ne regarde que les champs issus de la source. Les colonnes calculees par le
# pipeline (scores, drapeaux) sont remplies a 100 % par construction : les
# afficher noierait le signal.
CHAMPS_SOURCE = [
    "title", "company", "location", "contract_type", "education_required",
    "skills_required", "skills_optional", "start_date", "end_date", "duration",
    "remuneration", "published_at",
]

tableau_completude = COMPLETUDE_PAR_SOURCE.loc[CHAMPS_SOURCE, ORDRE_SOURCES].copy()
tableau_completude.insert(0, "ensemble", COMPLETUDE_GLOBALE.loc[CHAMPS_SOURCE])
tableau_completude.columns = ["Ensemble"] + [LIBELLE_SOURCE[s] for s in ORDRE_SOURCES]

def verdict(taux):
    if taux == 0:
        return "inutilisable (vide)"
    if taux < 15:
        return "inutilisable (trop rare)"
    if taux < 60:
        return "partiel - biais de selection"
    if taux < 100:
        return "exploitable"
    return "complet"

tableau_completude["Verdict"] = [verdict(t) for t in tableau_completude["Ensemble"]]

print("TAUX DE REMPLISSAGE (%) PAR CHAMP ET PAR SOURCE\n")
print(tableau_completude.to_string())

In [ ]:
LIBELLE_CHAMP = {
    "title": "Intitule du poste", "company": "Entreprise", "location": "Lieu",
    "contract_type": "Type de contrat", "education_required": "Niveau d'etudes",
    "skills_required": "Competences requises",
    "skills_optional": "Competences optionnelles",
    "start_date": "Date de debut", "end_date": "Date de fin",
    "duration": "Duree du contrat", "remuneration": "Remuneration",
    "published_at": "Date de publication",
}

fig, ax = plt.subplots(figsize=(9.4, 7.4))

champs = tableau_completude.index.tolist()[::-1]
y = np.arange(len(champs)) * 1.12   # respiration entre groupes, pas dans le groupe
hauteur = 0.30

for i, src in enumerate(ORDRE_SOURCES):
    valeurs = tableau_completude.loc[champs, LIBELLE_SOURCE[src]].to_numpy(dtype=float)
    # Decalage vertical + hauteur legerement reduite : la separation se fait par
    # un filet de fond, jamais par une bordure dessinee autour des barres.
    barres = ax.barh(y + (1 - i) * hauteur, valeurs, height=hauteur - 0.05,
                     color=COULEUR_SOURCE[src], label=LIBELLE_SOURCE[src],
                     linewidth=0, zorder=3)
    etiqueter(ax, barres, suffixe=" %")

ax.set_yticks(y)
ax.set_yticklabels([LIBELLE_CHAMP.get(c, c) for c in champs])
ax.set_ylim(-0.75, y.max() + 0.75)
ax.set_xlim(0, 116)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel("Champ renseigne (% des offres de la source)")
habiller(
    ax,
    "Chaque source est muette sur des champs differents",
    "Le vide n'est pas aleatoire : il dessine le contrat de donnees de chaque fournisseur",
)
# Legende sous l'axe : a l'interieur elle recouvrirait les barres a 100 %.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncols=3)
enregistrer(fig, "01_completude_par_source")
plt.show()

**Lecture.** Les trois sources ont des contrats de données nettement différents,
et aucune n'est bonne partout.

- **Les deux API** remplissent **toujours** le lieu et le type de contrat
  (100 %), et France Travail renseigne aussi la durée (100 %) et les compétences
  (84 %). Mais elles perdent souvent l'entreprise : 72 % pour France Travail,
  **58 % seulement** pour La Bonne Alternance.
- **LinkedIn via alertes mail** est la seule source qui identifie toujours
  l'entreprise (100 %) — c'est le point fort du mail d'alerte. En revanche elle
  ne fournit le type de contrat que dans **49 %** des cas, le lieu dans 82 %, et
  **jamais** la durée ni la date de début. L'information est extraite d'un corps
  de mail, pas d'un champ structuré : c'est la cause directe des parties 4 et 6.
- **Trois champs sont vides ou quasi vides partout** : compétences optionnelles
  (0 %), date de fin (0 %), date de début (4 %).

**Un piège de lecture à signaler.** La colonne `Verdict` juge le taux
*d'ensemble*, qui est dominé par LinkedIn (232 offres sur 276). `duration` y est
classée « inutilisable » à 14 % alors qu'elle est **complète sur France Travail**.
Un champ peut donc être exploitable *au sein d'une source* et inutilisable sur le
jeu entier. C'est ce que le croisement par source sert à rendre visible : une
moyenne globale de complétude aurait masqué les deux informations.

**Conséquence assumée** : les volets salaire, compétences et niveau d'études
annoncés comme exclus en introduction le sont pour cette raison mesurée, et le
sujet est clos ici. Analyser 31 rémunérations en texte libre, dont 0 venant de
La Bonne Alternance, aurait produit un chiffre, pas une information.

---
## 4. Couverture — qui ramène quoi, et où

### Pourquoi `zone inconnue` reste une catégorie

Les 51 offres sans zone ne sont **pas** supprimées. Les jeter ferait remonter
mécaniquement la part de Toulouse et de l'Île-de-France, et donnerait une
couverture artificiellement propre : la zone manquante est précisément l'un des
défauts de couverture que ce notebook est censé mesurer. Les écarter reviendrait
à effacer le résultat pour embellir le graphique.

Deux mécanismes distincts se cachent derrière cette catégorie, et le second est
un bug de parsing que seule cette décomposition fait apparaître.

In [ ]:
inconnues = offres[offres["zone"] == "inconnu"]
sans_lieu = int((~inconnues["location_renseignee"]).sum())
avec_lieu = int(inconnues["location_renseignee"].sum())

decomposition = pd.DataFrame({
    "offres": [sans_lieu, avec_lieu],
    "mecanisme": [
        "aucun lieu fourni par la source",
        "lieu fourni mais non reconnu comme une ville",
    ],
}, index=["champ lieu vide", "champ lieu non geolocalisable"])

print("DECOMPOSITION DES {} OFFRES EN ZONE INCONNUE\n".format(len(inconnues)))
print(decomposition.to_string())

print("\n\nRepartition par source de ces offres :")
print(inconnues["source"].value_counts().rename(LIBELLE_SOURCE).to_string())

print("\n\nLes {} lieux non geolocalisables viennent tous de la meme source.".format(avec_lieu))
print("Inspection du contenu (sans afficher les valeurs, qui sont nominatives) :")
print("  -> le champ 'lieu' contient en realite un nom d'entreprise comportant")
print("     le mot 'France' (societes nationales, filiales France de groupes).")
print("  -> le parseur des alertes mail confond le libelle entreprise et le")
print("     libelle lieu quand le premier ressemble a un toponyme.")
print("\nC'est un bug de parsing identifie, pas une donnee manquante a la source :")
print("l'information existe dans le mail, elle est rangee dans la mauvaise colonne.")

In [ ]:
couverture = pd.crosstab(offres["zone"], offres["source"])
couverture = couverture.reindex(index=ORDRE_ZONES, columns=ORDRE_SOURCES, fill_value=0)

couverture_affichee = couverture.rename(index=LIBELLE_ZONE, columns=LIBELLE_SOURCE)
couverture_affichee["Total"] = couverture_affichee.sum(axis=1)
couverture_affichee.loc["Total"] = couverture_affichee.sum()

print("COUVERTURE : OFFRES PAR SOURCE ET PAR ZONE (effectifs)\n")
print(couverture_affichee.to_string())

print("\n\nPart de chaque source dans sa zone (%) :\n")
part = (100 * couverture.T / couverture.sum(axis=1)).T.round(1)
print(part.rename(index=LIBELLE_ZONE, columns=LIBELLE_SOURCE).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 3.9))

zones = [z for z in ORDRE_ZONES if couverture.loc[z].sum() > 0][::-1]
y = np.arange(len(zones))
hauteur = 0.24

for i, src in enumerate(ORDRE_SOURCES):
    valeurs = couverture.loc[zones, src].to_numpy(dtype=float)
    barres = ax.barh(y + (1 - i) * hauteur, valeurs, height=hauteur - 0.035,
                     color=COULEUR_SOURCE[src], label=LIBELLE_SOURCE[src],
                     linewidth=0, zorder=3)
    etiqueter(ax, barres)

ax.set_yticks(y)
ax.set_yticklabels([LIBELLE_ZONE[z] for z in zones])
ax.set_xlim(0, couverture.to_numpy().max() * 1.16)
ax.set_xlabel("Nombre d'offres collectees (8 jours)")
habiller(
    ax,
    "Hors Ile-de-France, une seule source alimente la veille",
    "Les deux API n'ont ramene que 2 offres sur 78 a Toulouse",
)
ax.legend(loc="lower right")
enregistrer(fig, "02_couverture_source_zone")
plt.show()

In [ ]:
def tester_independance(table, libelle, seuil_effectif=5):
    '''Chi2 d'independance + V de Cramer, avec garde-fou sur les effectifs.

    Le chi2 n'est valide que si les effectifs theoriques sont suffisants. Quand
    ils ne le sont pas, deux cas :
      - table 2x2  -> on bascule sur le test exact de Fisher, qui n'a pas cette
                      contrainte et repond exactement a la meme question ;
      - plus grand -> la fonction refuse de conclure, plutot que de rendre une
                      p-valeur qui ne veut rien dire.
    '''
    tab = table.loc[table.sum(axis=1) > 0, table.sum(axis=0) > 0]
    chi2, p, dof, attendus = chi2_contingency(tab)
    n = int(tab.to_numpy().sum())
    v = float(np.sqrt(chi2 / (n * (min(tab.shape) - 1))))
    cellules_faibles = int((attendus < seuil_effectif).sum())
    methode = "chi2"

    print("TEST D'INDEPENDANCE - {}".format(libelle))
    print("  table {}x{}, n = {}".format(tab.shape[0], tab.shape[1], n))
    print("  chi2 = {:.1f}  ddl = {}  p = {:.2e}".format(chi2, dof, p))
    print("  V de Cramer = {:.2f}".format(v))

    if cellules_faibles and tab.shape == (2, 2):
        _, p = fisher_exact(tab.to_numpy())
        methode = "Fisher exact"
        print("  /!\\ {} cellule(s) d'effectif theorique < {} : le chi2 n'est pas".format(
            cellules_faibles, seuil_effectif))
        print("      valide. Bascule sur le test exact de Fisher -> p = {:.2e}".format(p))
    elif cellules_faibles:
        print("  /!\\ {} cellule(s) d'effectif theorique < {} : le test n'est pas".format(
            cellules_faibles, seuil_effectif))
        print("      valide sur cette table, et elle est trop grande pour Fisher.")
        print("      AUCUNE CONCLUSION n'en est tiree.")
        return {"libelle": libelle, "methode": "aucune", "n": n, "p": p, "v": v,
                "valide": False,
                "conclusion": "test non valide (effectifs theoriques insuffisants)"}

    if p < 0.05:
        force = "faible" if v < 0.2 else ("moderee" if v < 0.4 else "forte")
        conclusion = "lien detectable, intensite {} (V = {:.2f})".format(force, v)
        print("  => LIEN : p < 0.05 ({}). Intensite {} (V = {:.2f}).".format(
            methode, force, v))
    else:
        conclusion = "pas de lien detectable"
        print("  => PAS DE LIEN DETECTABLE : p >= 0.05 ({}).".format(methode))
        print("     Attention : 'pas de lien detectable' n'est pas 'pas de lien'.")
    print()
    return {"libelle": libelle, "methode": methode, "n": n, "p": p, "v": v,
            "valide": True, "conclusion": conclusion}


tests = []

# Premier essai : la table complete 3 sources x 4 zones. Elle contient des
# cellules a zero (aucune API hors Ile-de-France), donc des effectifs theoriques
# trop faibles. Le garde-fou doit refuser de conclure.
tests.append(tester_independance(
    pd.crosstab(offres["source"], offres["zone"]),
    "source x zone - table complete (3 x 4)",
))

Le test refuse de conclure, et c'est le comportement correct : la table complète
contient des cases vides (aucune des deux API n'a ramené une seule offre en zone
inconnue ni hors des deux zones cibles), ce qui écrase les effectifs théoriques
sous le seuil de validité du χ². Une p-valeur affichée ici aurait été lue comme
un résultat alors qu'elle n'en est pas un.

La question d'intérêt se teste sur une table plus resserrée, où chaque case est
suffisamment garnie : **sur les deux zones cibles, la source est-elle liée à la
zone ?** Les deux API sont regroupées, puisque l'enjeu est le contraste entre
« alertes mail » et « API structurées », pas entre les deux API.

In [ ]:
deux_zones = offres[offres["zone"].isin(["toulouse", "idf"])].copy()
deux_zones["canal"] = np.where(
    deux_zones["source"] == "linkedin", "Alertes mail", "API structurees"
)

table_canal = pd.crosstab(deux_zones["canal"], deux_zones["zone"])[["toulouse", "idf"]]
affiche_canal = table_canal.rename(columns=LIBELLE_ZONE)
affiche_canal["Total"] = affiche_canal.sum(axis=1)
print("CANAL DE COLLECTE x ZONE (zones cibles uniquement)\n")
print(affiche_canal.to_string())
print("\nPart des alertes mail dans chaque zone :")
for z in ["toulouse", "idf"]:
    tot = int(table_canal[z].sum())
    print("  {:15s} {:3d} / {:3d} = {:.0f} %".format(
        LIBELLE_ZONE[z], int(table_canal.loc["Alertes mail", z]), tot,
        100 * table_canal.loc["Alertes mail", z] / tot))
print()

tests.append(tester_independance(
    table_canal, "canal de collecte x zone (Toulouse / IDF)",
))

**Lecture de la couverture.**

Le déséquilibre est le résultat principal de cette partie : **LinkedIn ramène
97 % des offres toulousaines** (76 sur 78), tandis que les deux API, interrogées
avec les mêmes requêtes sur les deux villes, n'en ramènent que 2. En
Île-de-France elles en ramènent 42 sur 141, soit 30 %.

Le test sur la table resserrée détecte bien un lien entre le canal de collecte et
la zone, mais le chiffre important n'est pas la p-valeur : c'est qu'**une panne du
dépouillement des mails ferait disparaître la quasi-totalité de la visibilité sur
Toulouse**. La veille n'a, sur cette zone, aucune redondance — alors qu'en
Île-de-France trois sources se recoupent.

Le prix de cette source unique est visible juste à côté : **les 51 zones inconnues
viennent toutes de LinkedIn** — 41 parce que le mail ne porte pas de lieu
exploitable, 10 à cause du bug de parsing identifié plus haut. Autrement dit, la
seule source qui couvre Toulouse est aussi la seule qui perd l'information de
localisation.

---
## 5. Entonnoir de collecte, par source

`source_runs` journalise chaque exécution du collecteur. Le contrôle C1 a établi
que l'identité `trouvées = nouvelles + doublons + rejetées` tient exactement sur
les 554 runs : ces compteurs sont donc exploitables.

**Un point de lecture indispensable.** `offers_found` compte les offres *examinées*
à chaque passage, re-scans compris. Le collecteur repasse toutes les deux heures
sur les mêmes requêtes : une offre publiée lundi est donc comptée « trouvée » à
chaque run de la semaine. Les volumes bruts ne sont pas des volumes d'offres
distinctes, et les doublons sont majoritaires **par construction**, pas par
défaut de qualité. La lecture utile est donc *« sur 100 offres examinées,
combien sont nouvelles, déjà connues, écartées »*.

In [ ]:
entonnoir = runs.groupby("source_offre")[
    ["offers_found", "offers_new", "offers_duplicate", "offers_rejected"]
].sum()
entonnoir = entonnoir.reindex(ORDRE_SOURCES)
entonnoir.columns = ["examinees", "nouvelles", "doublons", "rejetees"]

entonnoir["runs"] = runs.groupby("source_offre").size().reindex(ORDRE_SOURCES)
entonnoir["runs_en_succes"] = (
    runs[runs["status"] == "success"].groupby("source_offre").size()
    .reindex(ORDRE_SOURCES).fillna(0).astype(int)
)
entonnoir["taux_succes_%"] = (
    100 * entonnoir["runs_en_succes"] / entonnoir["runs"]
).round(1)
entonnoir["nouvelles_par_run"] = (
    entonnoir["nouvelles"] / entonnoir["runs"]
).round(2)

print("ENTONNOIR DE COLLECTE (cumul sur 554 runs, 11 jours de journal)\n")
print(entonnoir.rename(index=LIBELLE_SOURCE).to_string())

print("\n\nPOUR 100 OFFRES EXAMINEES :\n")
pour_cent = entonnoir[["nouvelles", "doublons", "rejetees"]].div(
    entonnoir["examinees"], axis=0
) * 100
print(pour_cent.round(2).rename(index=LIBELLE_SOURCE).to_string())

In [ ]:
print("FIABILITE DES EXECUTIONS\n")
print(pd.crosstab(
    runs["source_offre"].map(LIBELLE_SOURCE), runs["status"], margins=True,
    margins_name="Total",
).to_string())

echecs = runs[runs["status"] != "success"]
print("\n{} run(s) non pleinement reussi(s) sur {} ({:.1f} %).".format(
    len(echecs), len(runs), 100 * len(echecs) / len(runs)))

if len(echecs):
    # On extrait le motif technique (code HTTP, timeout) sans reafficher les
    # requetes elles-memes, qui n'apportent rien ici.
    import re

    MOTIFS_CONNUS = [
        (r"HTTP 4?29", "quota depasse (HTTP 429)"),
        (r"HTTP 500|codeHttp.{0,3}500", "erreur serveur (HTTP 500)"),
        (r"timed out|timeout", "delai de lecture depasse"),
        (r"injoignable", "API injoignable"),
    ]
    print("\nNature des incidents :")
    for _, r in echecs.iterrows():
        texte = str(r["error"])
        motifs = [lib for motif, lib in MOTIFS_CONNUS if re.search(motif, texte)]
        print("  {} / {} -> {}".format(
            r["started_at"].strftime("%d/%m %H:%M"), r["source"],
            ", ".join(motifs) if motifs else "motif non reconnu"))
    print("\nLes deux incidents touchent la meme source, et aucun ne vient du")
    print("code de collecte : ce sont des defaillances ou des limites cote")
    print("fournisseur. Sur 186 appels, l'API France Travail en a honore 184.")

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 3.2))

sources = ORDRE_SOURCES[::-1]
y = np.arange(len(sources))
parts = pour_cent.loc[sources]

segments = [
    # Le segment "nouvelles" pese moins de 0,5 % : il est sous le pixel a cette
    # echelle. On le laisse dans la pile pour que les parts somment a 100, et on
    # le dit dans la legende plutot que de laisser croire a une couleur absente.
    ("nouvelles", BLEU, "Nouvelles (< 0,5 %, invisible ici)"),
    ("doublons", GRIS_NEUTRE, "Deja connues (re-scan)"),
    ("rejetees", ORANGE, "Ecartees par les filtres"),
]

gauche = np.zeros(len(sources))
for colonne, couleur, libelle in segments:
    valeurs = parts[colonne].to_numpy(dtype=float)
    ax.barh(y, valeurs, left=gauche, height=0.46, color=couleur, label=libelle,
            linewidth=1.6, edgecolor="#fcfcfb", zorder=3)
    for yi, (g, v) in enumerate(zip(gauche, valeurs)):
        # Etiquette dans le segment seulement si elle y tient : sinon elle serait
        # rognee, et la valeur exacte reste dans le tableau ci-dessus.
        if v >= 9:
            ax.text(g + v / 2, yi, "{:.0f} %".format(v), ha="center", va="center",
                    fontsize=8.8, color="#fcfcfb", fontweight="semibold", zorder=4)
    gauche = gauche + valeurs

for yi, src in enumerate(sources):
    ax.text(101.5, yi,
            "{:,} examinees  ->  {} nouvelles".format(
                int(entonnoir.loc[src, "examinees"]),
                int(entonnoir.loc[src, "nouvelles"])).replace(",", " "),
            va="center", ha="left", fontsize=8.6, color=ENCRE_2)

ax.set_yticks(y)
ax.set_yticklabels([LIBELLE_SOURCE[s] for s in sources])
ax.set_xlim(0, 100)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel("Part des offres examinees (%)")
habiller(
    ax,
    "La Bonne Alternance ecarte 3 offres sur 4 avant insertion",
    "Les doublons sont attendus : le collecteur repasse sur les memes requetes toutes les 2 h",
)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.34), ncols=3)
enregistrer(fig, "03_entonnoir_par_source")
plt.show()

**Lecture de l'entonnoir.**

- **Rendement très faible pour les API** : 0,18 offre nouvelle par run pour
  La Bonne Alternance, 0,12 pour France Travail — soit une offre utile tous les
  six à huit passages. Le dépouillement des mails fait mieux (0,96 par run) parce
  qu'un mail n'arrive que s'il y a quelque chose à annoncer. Interroger les deux
  API toutes les deux heures ne se justifie pas par le flux observé : c'est un
  résultat actionnable sur le dimensionnement de la collecte.
- **Les profils de filtrage sont opposés** : La Bonne Alternance écarte 75 % de
  ce qu'elle renvoie (son index est très large et peu ciblé), France Travail
  31 %, le dépouillement des mails 27 %. Une source généreuse n'est pas une
  meilleure source.
- **Les exécutions sont fiables** : 552 runs sur 554 en succès complet. Les deux
  incidents viennent de l'API France Travail (timeout, HTTP 500, quota HTTP 429),
  côté fournisseur.

Attention à ne pas confondre les deux rejets du pipeline : celui compté ici est le
**filtrage avant insertion** (l'offre n'entre pas en base). La partie 6 porte sur
le **statut attribué après insertion** aux offres conservées. Ce sont deux étapes
différentes, avec des motifs différents.

---
## 6. Motifs de rejet — le reclassement nécessaire

Telle quelle, la colonne `rejection_category` affirme que **120 des 132 offres
rejetées l'ont été pour « éligibilité »**. Reprise sans vérification, cette
étiquette donnerait la conclusion suivante : *« 91 % des offres rejetées ne
correspondent pas à mon profil »*.

Cette conclusion serait fausse. Le détail du journal de décision montre que la
quasi-totalité de ces rejets porte la mention `type de contrat incompatible :
None` — c'est-à-dire que **la source n'a fourni aucun type de contrat**. Le moteur
a traité l'absence d'information comme une incompatibilité.

La distinction n'est pas cosmétique :

- « offre inéligible » ⇒ *le marché ne propose pas ce que je cherche* ;
- « type de contrat non fourni » ⇒ *la source ne dit pas ce que l'offre propose*.

Le premier est un constat sur le marché, le second un défaut de métadonnée. Le
reclassement ci-dessous sépare les deux.

In [ ]:
def motif_reclasse(ligne):
    '''Motif de rejet reconstruit a partir du journal de decision.

    L'ordre des tests compte : une offre peut cumuler plusieurs mentions, et on
    retient la cause la plus specifique (une ecole imposee est un refus ferme,
    quel que soit l'etat des autres champs).
    '''
    if ligne["status"] != "rejected":
        return pd.NA
    if ligne["ecole_imposee"]:
        return "Ecole / CFA partenaire impose"
    if ligne["cdi_cdd_mal_classe"]:
        return "CDI-CDD mal classe par la source"
    if ligne["ecarte_manuellement"]:
        return "Ecarte manuellement"
    if ligne["contrat_non_fourni"]:
        return "Type de contrat non fourni par la source"
    return "Criteres de date ou de duree"


offres["motif_rejet"] = offres.apply(motif_reclasse, axis=1)

rejetees = offres[offres["status"] == "rejected"]

comparaison = pd.DataFrame({
    "avant (rejection_category)": rejetees["rejection_category"].value_counts(),
    "apres reclassement": rejetees["motif_rejet"].value_counts(),
}).fillna(0).astype(int)

print("{} offres rejetees sur {} ({:.0f} %)\n".format(
    len(rejetees), len(offres), 100 * len(rejetees) / len(offres)))
print("AVANT / APRES RECLASSEMENT\n")
print(comparaison.to_string())

GAP = int((rejetees["motif_rejet"] == "Type de contrat non fourni par la source").sum())
# "eligibilite" est accentue dans les donnees : on cible le radical pour ne pas
# attraper aussi "ecole imposee", qui commence par la meme lettre accentuee.
LOT_ELIGIBILITE = int(
    rejetees["rejection_category"].str.contains("ligibilit", na=False).sum()
)
print("\n\n-> {} des {} rejets etiquetes 'eligibilite' sont en realite une".format(
    GAP, LOT_ELIGIBILITE))
print("   metadonnee absente, pas une offre inadaptee a mon profil ({:.0f} %).".format(
    100 * GAP / LOT_ELIGIBILITE))
print("\nVERITABLES rejets sur criteres de profil : {} offres sur {}, soit {:.1f} %.".format(
    int((rejetees["motif_rejet"] == "Criteres de date ou de duree").sum()),
    len(offres),
    100 * (rejetees["motif_rejet"] == "Criteres de date ou de duree").sum() / len(offres),
))

In [ ]:
print("MOTIF DE REJET RECLASSE, PAR SOURCE (effectifs)\n")
motifs_source = pd.crosstab(rejetees["motif_rejet"], rejetees["source"])
motifs_source = motifs_source.reindex(
    columns=[s for s in ORDRE_SOURCES if s in motifs_source.columns], fill_value=0
)
affiche = motifs_source.rename(columns=LIBELLE_SOURCE)
affiche["Total"] = affiche.sum(axis=1)
print(affiche.sort_values("Total", ascending=False).to_string())

# La table 5 x 3 est trop creuse pour un chi2 (beaucoup de motifs a un ou deux
# cas). L'affirmation a tester est plus simple et tient dans un tableau 2 x 2 :
# le trou de metadonnee est-il propre aux alertes mail ?
binaire = pd.crosstab(
    np.where(
        rejetees["motif_rejet"] == "Type de contrat non fourni par la source",
        "Type de contrat non fourni", "Autre motif",
    ),
    np.where(rejetees["source"] == "linkedin", "Alertes mail", "API structurees"),
)
print("\n\nLE TROU DE METADONNEE EST-IL PROPRE AUX ALERTES MAIL ?\n")
print(binaire.to_string())
print()

tests.append(tester_independance(
    binaire, "motif 'contrat non fourni' x canal de collecte",
))

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 3.4))

serie = rejetees["motif_rejet"].value_counts().sort_values()
MOTIF_CLE = "Type de contrat non fourni par la source"

# Emphase : le motif qui porte le resultat garde la couleur d'accent, les autres
# passent en gris. Un degrade par valeur serait un double encodage de la
# longueur de barre.
couleurs = [BLEU if m == MOTIF_CLE else GRIS_NEUTRE for m in serie.index]
barres = ax.barh(np.arange(len(serie)), serie.to_numpy(dtype=float), height=0.56,
                 color=couleurs, linewidth=0, zorder=3)
ax.set_xlim(0, serie.max() * 1.22)
etiqueter(ax, barres)

ax.set_yticks(np.arange(len(serie)))
ax.set_yticklabels(serie.index)
ax.set_xlabel("Nombre d'offres rejetees")
habiller(
    ax,
    "Le premier motif de rejet est une metadonnee absente, pas un critere de profil",
    "Apres reclassement des 120 rejets etiquetes << eligibilite >>",
)
enregistrer(fig, "04_motifs_rejet_reclasses")
plt.show()

**Lecture.**

Le motif dominant n'est pas un désaccord avec le marché, c'est un **trou de
métadonnée** : 115 offres rejetées sur 132 l'ont été parce qu'aucun type de
contrat n'était fourni. Toutes viennent de LinkedIn, et la partie 3 avait déjà
mesuré la cause — `contract_type` n'est renseigné que sur la moitié des offres
issues des alertes mail.

À l'inverse, **3 offres seulement** sur 276 ont été écartées sur un véritable
critère de profil (date de début trop tardive, durée de 6 ou de 36 mois). Le
chiffre honnête à retenir est donc : *« le filtrage automatique écarte 48 % du
flux, et pour 87 % de ces pertes la cause est une information manquante côté
source, pas une inadéquation au profil »*.

Le test sur le tableau 2 × 2 confirme que ce trou de métadonnée est **entièrement
concentré sur les alertes mail** : les deux API n'y contribuent pas une seule
offre.

C'est un résultat exploitable : la bonne action n'est pas d'assouplir les critères
du moteur, c'est de **récupérer le type de contrat** — en consultant la page de
l'offre plutôt que le seul corps du mail. Le gisement est de 115 offres sur 8 jours.

> Ce reclassement est fait **ici, dans le notebook**, à partir du journal de
> décision — il n'a pas été corrigé en amont dans la base. La colonne d'origine
> `rejection_category` est conservée et affichée à côté, pour que l'écart reste
> visible et vérifiable.

---
## 7. Familles de métiers par zone

La famille de métier est déduite de l'intitulé par le pipeline. Deux précautions
avant de lire ce tableau :

1. **La règle des effectifs.** Toute famille dont l'effectif total sur les deux
   zones est inférieur à 30 est signalée et **aucune conclusion n'en est tirée** :
   à cette taille, un écart de deux offres déplace une part de plusieurs points.
2. **Le biais de requête.** Cette répartition reflète mes requêtes de recherche
   autant que les offres existantes. Les deux API sont interrogées avec la même
   liste de métiers pour les deux villes, ce qui limite l'asymétrie ; en revanche
   la composition des alertes mail LinkedIn — qui fournit 97 % des offres
   toulousaines — n'est pas sous mon contrôle. Un écart de répartition entre
   Toulouse et l'Île-de-France peut donc venir du marché **ou** du réglage des
   alertes, et ce jeu de données ne permet pas de trancher.

In [ ]:
SEUIL_N = 30

metiers_zone = pd.crosstab(offres["role_family"], offres["zone"])
metiers_zone = metiers_zone.reindex(
    columns=[z for z in ORDRE_ZONES if z in metiers_zone.columns], fill_value=0
)

ZONES_ETUDE = ["toulouse", "idf"]
etude = metiers_zone[ZONES_ETUDE].copy()
etude["total_2_zones"] = etude.sum(axis=1)
etude["effectif_suffisant"] = etude["total_2_zones"] >= SEUIL_N
etude = etude.sort_values("total_2_zones", ascending=False)

affiche = etude.rename(index=LIBELLE_METIER)
affiche.columns = ["Toulouse", "Ile-de-France", "Total", "n >= {}".format(SEUIL_N)]
print("FAMILLES DE METIERS PAR ZONE (effectifs)\n")
print(affiche.to_string())

print("\n\nParts au sein de chaque zone (%) :\n")
parts_zone = (100 * etude[ZONES_ETUDE] / etude[ZONES_ETUDE].sum()).round(1)
print(parts_zone.rename(index=LIBELLE_METIER).rename(
    columns={"toulouse": "Toulouse", "idf": "Ile-de-France"}).to_string())

insuffisants = etude.index[~etude["effectif_suffisant"]].tolist()
print("\n\nFAMILLES A EFFECTIF INSUFFISANT (n < {}) : {}".format(
    SEUIL_N, ", ".join(LIBELLE_METIER[m] for m in insuffisants)))
for m in insuffisants:
    print("  {:18s} n = {:2d} (Toulouse {}, IDF {}) -> aucune conclusion".format(
        LIBELLE_METIER[m], int(etude.loc[m, "total_2_zones"]),
        int(etude.loc[m, "toulouse"]), int(etude.loc[m, "idf"])))

In [ ]:
# Le test ne porte que sur les familles a effectif suffisant. Inclure DevOps
# (n=12, dont 0 a Toulouse) et Mobile (n=2) ferait tomber des effectifs
# theoriques sous 5 et invaliderait le chi2 sur l'ensemble de la table.
robustes = etude.index[etude["effectif_suffisant"]].tolist()
table_test = metiers_zone.loc[robustes, ZONES_ETUDE]

print("Familles retenues pour le test : {}\n".format(
    ", ".join(LIBELLE_METIER[m] for m in robustes)))
tests.append(tester_independance(
    table_test.T, "famille de metier x zone (familles n >= {} seulement)".format(SEUIL_N),
))

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 3.8))

familles = etude.index.tolist()[::-1]
y = np.arange(len(familles))
hauteur = 0.34

for i, zone in enumerate(ZONES_ETUDE):
    valeurs = etude.loc[familles, zone].to_numpy(dtype=float)
    barres = ax.barh(y + (0.5 - i) * hauteur, valeurs, height=hauteur - 0.045,
                     color=COULEUR_ZONE[zone], label=LIBELLE_ZONE[zone],
                     linewidth=0, zorder=3)
    etiqueter(ax, barres)

etiquettes = [
    LIBELLE_METIER[m] + ("" if etude.loc[m, "effectif_suffisant"] else "  (n < 30)")
    for m in familles
]
ax.set_yticks(y)
ax.set_yticklabels(etiquettes)
ax.set_xlim(0, etude[ZONES_ETUDE].to_numpy().max() * 1.2)
ax.set_xlabel("Nombre d'offres collectees (8 jours)")
habiller(
    ax,
    "Composition des deux zones : data en Ile-de-France, full stack a Toulouse",
    "Les familles marquees (n < 30) sont affichees mais ne sont pas interpretees",
)
ax.legend(loc="lower right")
enregistrer(fig, "05_metiers_par_zone")
plt.show()

**Lecture.**

Trois familles atteignent l'effectif requis : full stack (68), data (64) et la
catégorie fourre-tout « autre » (48). Sur ces trois, la composition des deux
zones s'inverse : **la data pèse 37 % des offres franciliennes collectées contre
15 % des toulousaines**, et le full stack 42 % à Toulouse contre 25 % en
Île-de-France.

Le test confirme un lien statistique, mais la limite énoncée plus haut reste
décisive : **on ne peut pas attribuer cet écart au marché**, puisque la source
qui porte presque toute la collecte toulousaine est une alerte mail dont la
composition thématique n'est pas mesurable depuis ce jeu de données. La phrase
défendable est : *« ma veille ramène un mix différent selon la zone »*, pas
*« Toulouse recrute moins en data »*.

Backend (n = 26, réparti 13 / 13), DevOps (n = 12, dont aucune offre toulousaine)
et mobile (n = 1) sont affichés pour ne rien cacher du volume réel, et
explicitement exclus du test comme de toute conclusion. Le cas de DevOps est
tentant — zéro offre à Toulouse se lit volontiers comme « Toulouse ne recrute pas
de DevOps en alternance » — et c'est précisément pour ce genre de phrase que le
seuil existe : sur 12 observations, cet écart n'est pas distinguable du hasard
d'échantillonnage.

---
## 8. Délais de détection — diagnostic avant interprétation

La table propose `detection_delay_days` et `age_at_detection_days`, qui semblent
mesurer la réactivité de la veille : *combien de temps après sa publication une
offre est-elle vue ?*

Avant d'en tirer le moindre chiffre, trois vérifications s'imposent. Les trois
échouent, et le résultat de cette partie est donc **négatif** : la réactivité
n'est pas mesurable sur ce snapshot. C'est une conclusion, pas une absence de
conclusion.

In [ ]:
# --- Verification 1 : des delais negatifs ---------------------------------
# Un delai negatif signifie une offre vue avant d'etre publiee. C'est impossible,
# donc c'est un defaut de donnee : le signaler et l'ecarter, jamais le garder.
negatifs = offres[offres["detection_delay_days"] < 0]
print("VERIFICATION 1 - delais negatifs\n")
print("{} offre(s) avec un delai de detection negatif.".format(len(negatifs)))
if len(negatifs):
    print("Amplitude : de {:.2f} a {:.2f} jour ({:.0f} a {:.0f} minutes).".format(
        negatifs["detection_delay_days"].min(), negatifs["detection_delay_days"].max(),
        negatifs["detection_delay_days"].min() * 1440,
        negatifs["detection_delay_days"].max() * 1440))
    print("Source(s) concernee(s) :", ", ".join(
        LIBELLE_SOURCE[s] for s in negatifs["source"].unique()))
    print("\nAmplitude inferieure a deux heures et une seule source touchee :")
    print("signature d'un decalage de fuseau ou d'horloge entre l'API et le")
    print("serveur de collecte, pas d'une erreur de calcul du delai.")
    print("-> ces {} lignes sont ECARTEES de tout calcul de delai.".format(len(negatifs)))

In [ ]:
# --- Verification 2 : published_at de LinkedIn est-elle une vraie date de
#     publication, ou la date d'arrivee du mail d'alerte ? ------------------
print("VERIFICATION 2 - provenance de published_at\n")

diag = offres.dropna(subset=["published_at"]).copy()
diag["minute"] = diag["published_at"].dt.minute
diag["ecart_min"] = (
    (diag["first_seen_at"] - diag["published_at"]).dt.total_seconds() / 60
)

resume = pd.DataFrame({
    "offres": diag.groupby("source").size(),
    "minutes_distinctes": diag.groupby("source")["minute"].nunique(),
    "part_a_la_minute_10_%": (
        100 * diag.groupby("source")["minute"].apply(lambda s: (s == 10).mean())
    ).round(1),
    "published_at_le_plus_ancien": diag.groupby("source")["published_at"].min().dt.date,
    "ecart_median_min": diag.groupby("source")["ecart_min"].median().round(1),
}).reindex(ORDRE_SOURCES)

print(resume.rename(index=LIBELLE_SOURCE).to_string())

print("\n\nEcarts les plus frequents entre published_at et first_seen_at (LinkedIn) :\n")
li = diag[diag["source"] == "linkedin"]
ecarts = li["ecart_min"].round(0)
for ecart, n in ecarts.value_counts().head(5).items():
    marque = "  <- intervalle mail -> collecteur" if ecart <= 60 else ""
    print("  {:>7.0f} min  ->  {:3d} offres{}".format(ecart, n, marque))

SOUS_UNE_HEURE = int((ecarts <= 60).sum())
print("\n  {} offres sur {} ({:.0f} %) sont vues moins de 61 minutes apres leur".format(
    SOUS_UNE_HEURE, len(li), 100 * SOUS_UNE_HEURE / len(li)))
print("  pretendue publication. Aucune offre LinkedIn n'a plus de {:.0f} jours.".format(
    ecarts.max() / 1440))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9.2, 5.0), sharex=True)

for ax, src in zip(axes, ORDRE_SOURCES):
    sous = diag[diag["source"] == src]
    comptes = sous["minute"].value_counts().reindex(range(60), fill_value=0)
    ax.bar(comptes.index, comptes.to_numpy(dtype=float), width=0.72,
           color=COULEUR_SOURCE[src], linewidth=0, zorder=3)
    ax.set_ylabel("offres", fontsize=8.5)
    ax.grid(axis="y", alpha=0.9, zorder=0)
    ax.set_axisbelow(True)
    # Des offres se comptent en entiers : pas de graduation a 0,5.
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, nbins=3))
    ax.text(0.995, 0.88, "{}  (n = {})".format(LIBELLE_SOURCE[src], len(sous)),
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            color=ENCRE, fontweight="semibold")
    pic = comptes.idxmax()
    if comptes.loc[pic] > 0.5 * comptes.sum():
        ax.annotate(
            "{} offres a la minute :{:02d}".format(int(comptes.loc[pic]), pic),
            xy=(pic, comptes.loc[pic]), xytext=(pic + 6, comptes.loc[pic] * 0.92),
            fontsize=8.6, color=ENCRE_2, va="center",
            arrowprops=dict(arrowstyle="-", color="#b9b7b0", linewidth=0.8),
        )

axes[-1].set_xlabel("Minute de l'heure portee par published_at")
axes[-1].set_xlim(-1, 60)
axes[-1].set_xticks([0, 10, 20, 30, 40, 50, 59])
axes[0].set_title("La date de publication LinkedIn tombe presque toujours a la minute :10")
axes[0].text(0, 1.03,
             "Une vraie date de publication serait repartie sur les 60 minutes, comme celle des deux API",
             transform=axes[0].transAxes, ha="left", va="bottom",
             fontsize=8.6, color=ENCRE_2)
fig.align_ylabels(axes)
enregistrer(fig, "06_diagnostic_published_at")
plt.show()

### Verdict de la vérification 2

Trois indices convergents, et aucun n'est compatible avec une vraie date de
publication :

1. **L'horodatage est quantifié.** Les `published_at` de LinkedIn ne prennent que
   **9 valeurs de minute distinctes sur 60**, et **87 %** tombent exactement à la
   minute `:10`. Les deux API, elles, s'étalent sur 17 et 18 minutes distinctes
   pour seulement 25 et 18 offres. Une horloge aussi régulière est celle d'un
   **envoi programmé** — une alerte qui part toutes les deux heures à HH:10 — pas
   celle de centaines d'employeurs publiant leurs annonces.
2. **L'écart avec la détection est constant.** 101 offres présentent exactement
   **51 minutes** d'écart entre `published_at` et `first_seen_at`, et 134 sur 232
   (58 %) tiennent sous l'heure. C'est l'intervalle entre l'arrivée du mail et le
   passage suivant du collecteur, qui tourne à l'heure ronde.
3. **L'historique s'arrête net.** Le `published_at` LinkedIn le plus ancien est le
   22 septembre, soit le début de l'historique de la boîte mail — alors que
   La Bonne Alternance remonte au 18 mai et France Travail au 21 juillet.
   LinkedIn ne peut pas « ne rien avoir publié » avant le 22 septembre.

**`published_at` de LinkedIn est l'heure d'arrivée du mail d'alerte, pas la date
de publication de l'offre.** Par conséquent `detection_delay_days` ne mesure, pour
LinkedIn, que **la latence interne du pipeline** (mail reçu → mail dépouillé) :
d'où sa médiane de 0,04 jour, soit 51 minutes.

> Sans ce diagnostic, la lecture naïve du tableau aurait donné : *« LinkedIn
> détecte les offres en 51 minutes contre 19 jours pour France Travail — LinkedIn
> est 537 fois plus réactif »*. Ce chiffre est un artefact de construction : il
> compare la latence d'une tâche planifiée à l'âge réel d'une annonce. C'est le
> genre d'erreur qu'une moyenne bien présentée rend invisible, et qu'aucun
> traitement statistique en aval n'aurait rattrapée.

In [ ]:
# --- Verification 3 : combien d'offres restent exploitables ? --------------
print("VERIFICATION 3 - echantillon reellement exploitable\n")

exploitable = offres.copy()
motifs_exclusion = {}

avant = len(exploitable)
exploitable = exploitable[exploitable["source"] != "linkedin"]
motifs_exclusion["published_at = date du mail (verification 2)"] = avant - len(exploitable)

avant = len(exploitable)
exploitable = exploitable[exploitable["detection_delay_days"].notna()]
motifs_exclusion["published_at absent, donc aucun delai calculable"] = (
    avant - len(exploitable)
)

avant = len(exploitable)
exploitable = exploitable[exploitable["detection_delay_days"] >= 0]
motifs_exclusion["delai negatif (verification 1)"] = avant - len(exploitable)

avant = len(exploitable)
exploitable = exploitable[~exploitable["first_seen_at"].isin(LOTS_MASSE.index)]
motifs_exclusion["first_seen_at = heure d'un import en masse (controle C5)"] = (
    avant - len(exploitable)
)

print("Depart : {} offres\n".format(len(offres)))
for motif, n in motifs_exclusion.items():
    print("  - {:3d}  {}".format(n, motif))
print("\nReste : {} offres exploitables pour mesurer un delai de detection.".format(
    len(exploitable)))

if len(exploitable):
    print("\nRepartition : {}".format(
        exploitable["source"].map(LIBELLE_SOURCE).value_counts().to_dict()))
    print("\nStatistiques indicatives (age de l'offre a la detection, en jours) :\n")
    print(exploitable.groupby("source")["age_at_detection_days"]
          .agg(n="size", mediane="median", minimum="min", maximum="max")
          .rename(index=LIBELLE_SOURCE).round(1).to_string())

print("\n" + "=" * 72)
print("n = {} << {} : SOUS LE SEUIL D'EFFECTIF. AUCUNE CONCLUSION N'EST TIREE.".format(
    len(exploitable), SEUIL_N))
print("=" * 72)

### Conclusion de la partie 8

Le résultat est négatif, et c'est le résultat : **la réactivité de la veille n'est
pas mesurable sur ce snapshot.** Le tableau de départ semblait fournir 275 délais
de détection ; après les trois vérifications il en reste **3**, très loin du seuil
de 30 fixé en partie 7. Les chiffres affichés ci-dessus sont indicatifs et ne sont
pas interprétés.

Le détail de l'attrition est instructif : 232 lignes tombent parce que leur date
de publication n'en est pas une, et 34 des 44 lignes restantes tombent parce que
leur date de détection est en réalité une heure de ré-import. Autrement dit, sur
les deux dates nécessaires au calcul, **chacune est corrompue pour une raison
différente**, et les deux corruptions touchent des populations presque
complémentaires.

**Ce que ce résultat négatif implique concrètement**, et c'est la partie
actionnable : la table confond trois horodatages qui doivent être distingués à
la collecte.

| Ce qu'il faut journaliser | Aujourd'hui | Signification |
|---|---|---|
| `published_at` | mélangé selon la source | date de publication **chez l'employeur** |
| `received_at` | absent | arrivée du mail ou réponse de l'API |
| `detected_at` | confondu avec `first_seen_at` | passage du collecteur |
| `first_seen_at` | écrasé lors des re-imports | première insertion en base |

Tant que ces quatre dates sont confondues en deux colonnes, et qu'un re-import
réécrit `first_seen_at`, aucun indicateur de réactivité ne sera fiable. Il faut
aussi conserver la **provenance** de `published_at` (champ fourni par la source /
déduit de l'enveloppe du mail) : c'est elle qui manquait ici et dont l'absence a
failli produire un faux résultat.

---
## 9. Export des agrégats

Seuls des **tableaux agrégés** sortent vers `data/public/` : aucune ligne
d'offre, donc aucun intitulé, entreprise ou URL. Un garde-fou vérifie cette
promesse plutôt que de la supposer — il refuse tout export dont le nombre de
lignes approcherait celui du jeu de données.

In [ ]:
PLAFOND_LIGNES = 40  # un export agrege n'a pas 276 lignes


def exporter(df, nom, index=True):
    '''Ecrit un agregat dans data/public/ apres verification anti-fuite.'''
    if len(df) > PLAFOND_LIGNES:
        raise ValueError(
            "{} fait {} lignes (> {}) : cela ressemble a un export ligne a ligne, "
            "pas a un agregat. Export refuse.".format(nom, len(df), PLAFOND_LIGNES)
        )
    interdites = set(COLONNES_SENSIBLES) & set(df.columns)
    if interdites:
        raise ValueError("{} contient des colonnes sensibles : {}".format(nom, interdites))

    chemin = DOSSIER_PUBLIC / (nom + ".csv")
    df.to_csv(chemin, index=index, encoding="utf-8")
    print("  {:38s} {:2d} lignes".format(chemin.name, len(df)))
    return chemin


print("Exports vers data/public/ :\n")

exporter(TABLEAU_CONTROLES, "01_controles_coherence", index=False)
exporter(tableau_completude, "02_completude_par_champ_et_source")
exporter(couverture_affichee, "03_couverture_source_zone")
exporter(decomposition, "04_decomposition_zone_inconnue")
exporter(entonnoir.rename(index=LIBELLE_SOURCE), "05_entonnoir_par_source")
exporter(comparaison, "06_motifs_rejet_avant_apres")
exporter(affiche_metiers := etude.rename(index=LIBELLE_METIER), "07_metiers_par_zone")
exporter(resume.rename(index=LIBELLE_SOURCE), "08_diagnostic_published_at")
exporter(pd.DataFrame(tests), "09_tests_independance", index=False)

print("\nTous les exports sont des agregats. Aucune offre n'est exportee ligne a ligne.")

---
## 10. Limites

**Sur la portée.**

- Échantillon de **276 offres** issu de **ma** recherche personnelle : deux zones,
  une liste de requêtes, un jeu d'alertes mail. Ce n'est pas un échantillon du
  marché, et aucun chiffre de ce notebook n'autorise une phrase sur « le marché
  français de l'alternance IT ».
- Fenêtre d'observation réelle de **8 jours** (26 septembre → 3 octobre 2026),
  établie par le contrôle C2 et non par la date de démarrage du pipeline.
  Huit jours ne permettent de voir ni la saisonnalité, ni les campagnes de
  recrutement de rentrée.
- Snapshot figé au **3 octobre 2026**. Une offre « à examiner » à cette date a pu
  être pourvue depuis.

**Sur les sources.**

- Les trois sources ne sont pas comparables terme à terme : deux API structurées
  et un dépouillement d'alertes mail. Le second n'offre aucune garantie de
  complétude, et sa composition thématique n'est pas sous contrôle.
- La couverture de Toulouse repose à 97 % sur une source unique. Toute conclusion
  « Toulouse vs Île-de-France » hérite de ce déséquilibre.
- `labonnealternance` et `france_travail` ont pu voir leur quota ou leur
  disponibilité varier pendant la période (deux incidents journalisés).

**Sur les mesures.**

- La famille de métier est **déduite de l'intitulé** par le pipeline, pas
  déclarée par l'employeur. La catégorie « autre » (57 offres) regroupe ce que la
  règle ne sait pas classer.
- `detection_delay_days` n'est pas interprété : voir partie 8. Les délais
  négatifs (6 offres) sont écartés, les lots d'import (133 offres) aussi.
- Les volumes de `source_runs` comptent les offres **examinées**, re-scans
  compris. Ils ne mesurent pas des offres distinctes.
- Les motifs de rejet de la partie 6 sont **reclassés dans le notebook** à partir
  du journal de décision. La base n'a pas été corrigée : la colonne d'origine est
  conservée et affichée à côté.

**Sur ce qui n'a pas pu être analysé.**

Salaire, compétences et niveau d'études sont hors périmètre faute de remplissage
(partie 3), pas par choix de cadrage. Si le pipeline récupérait la page de
l'offre plutôt que le seul corps du mail, ces trois volets deviendraient
analysables — et 115 rejets sur 132 disparaîtraient.

---

*Fanel BALEMO — L2 MIASHS, Université Toulouse III · snapshot du 3 octobre 2026*